In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/Multimodal/multibully_with_captions.csv")

print(f"Rows loaded        : {len(df)}")
print(f"Real captions      : {(~df['blip_caption'].str.startswith('ERROR')).sum()}")
print(f"Error captions     : {df['blip_caption'].str.startswith('ERROR').sum()}")
print(f"\nSample caption: {df['blip_caption'].iloc[0]}")

Rows loaded        : 5793
Real captions      : 5793
Error captions     : 0

Sample caption: a group of women with glasses on their faces


In [ ]:
# Replace ERROR captions with empty string

df['blip_caption'] = df['blip_caption'].apply(
    lambda x: "" if str(x).startswith("ERROR") else str(x)
)
df['blip_caption'] = df['blip_caption'].fillna("")

print(f"Captions cleaned")
print(f"Empty captions : {(df['blip_caption'] == '').sum()}")
print(f"Good captions  : {(df['blip_caption'] != '').sum()}")

Captions cleaned
Empty captions : 0
Good captions  : 5793


In [ ]:
# Template A — main template (caption + text, structured)
def build_prompt_A(caption, meme_text):
    return f"The image shows {caption}. The meme text says {meme_text}. This meme is <mask>."

# Template B — simpler version (caption + text, short)
def build_prompt_B(caption, meme_text):
    return f"{caption}. {meme_text}. Overall this is <mask>."

# Template C — caption only (no meme text)
# tests: does image caption alone help?
def build_prompt_C(caption, meme_text):
    return f"The image shows {caption}. The sentiment of this image is <mask>."

# Template D — text only (no caption at all)
# tests: does removing the image hurt performance?
# directly comparable to MultiBully text-only baseline (F1 ~61)
def build_prompt_D(caption, meme_text):
    return f"The meme text says {meme_text}. This meme is <mask>."

print("Four prompt templates defined:")
print("\nTemplate A (main - caption + text, structured):")
print('  "The image shows [caption]. The meme text says [text]. This meme is <mask>."')
print("\nTemplate B (ablation - caption + text, simple):")
print('  "[caption]. [text]. Overall this is <mask>."')
print("\nTemplate C (ablation - caption only, no text):")
print('  "The image shows [caption]. The sentiment of this image is <mask>."')
print("\nTemplate D (ablation - text only, no caption):")
print('  "The meme text says [text]. This meme is <mask>."')

Four prompt templates defined:

Template A (main - caption + text, structured):
  "The image shows [caption]. The meme text says [text]. This meme is <mask>."

Template B (ablation - caption + text, simple):
  "[caption]. [text]. Overall this is <mask>."

Template C (ablation - caption only, no text):
  "The image shows [caption]. The sentiment of this image is <mask>."

Template D (ablation - text only, no caption):
  "The meme text says [text]. This meme is <mask>."


In [ ]:
test_rows = df.sample(3, random_state=42)

print("TESTING ALL FOUR PROMPT TEMPLATES")
print("=" * 65)

for _, row in test_rows.iterrows():
    label   = "BULLY" if row["bully_label"] == 1 else "NOT BULLY"
    caption = row["blip_caption"]
    text    = str(row["Img-Text"])

    print(f"\n[{label}] {row['Img-Name']}")
    print(f"Caption   : {caption}")
    print(f"Meme text : {text[:60]}")
    print(f"\nPrompt A  : {build_prompt_A(caption, text)}")
    print(f"Prompt B  : {build_prompt_B(caption, text)}")
    print(f"Prompt C  : {build_prompt_C(caption, text)}")
    print(f"Prompt D  : {build_prompt_D(caption, text)}")
    print("-" * 65)

TESTING ALL FOUR PROMPT TEMPLATES

[BULLY] 1913.jpg
Caption   : a woman is getting her hair done
Meme text : This is bit weird

Prompt A  : The image shows a woman is getting her hair done. The meme text says This is bit weird. This meme is <mask>.
Prompt B  : a woman is getting her hair done. This is bit weird. Overall this is <mask>.
Prompt C  : The image shows a woman is getting her hair done. The sentiment of this image is <mask>.
Prompt D  : The meme text says This is bit weird. This meme is <mask>.
-----------------------------------------------------------------

[NOT BULLY] 1568.png
Caption   : a man and woman are laughing while they are singing
Meme text : when black people make white jokes when white people make bl

Prompt A  : The image shows a man and woman are laughing while they are singing. The meme text says when black people make white jokes when white people make black jokes. This meme is <mask>.
Prompt B  : a man and woman are laughing while they are singing. when bl

In [ ]:
print("Building prompts for all rows...")

df["prompt_A"] = df.apply(
    lambda row: build_prompt_A(row["blip_caption"], str(row["Img-Text"])),
    axis=1
)

df["prompt_B"] = df.apply(
    lambda row: build_prompt_B(row["blip_caption"], str(row["Img-Text"])),
    axis=1
)

df["prompt_C"] = df.apply(
    lambda row: build_prompt_C(row["blip_caption"], str(row["Img-Text"])),
    axis=1
)

df["prompt_D"] = df.apply(
    lambda row: build_prompt_D(row["blip_caption"], str(row["Img-Text"])),
    axis=1
)

print(f"Done — prompts built for {len(df)} rows")
print(f"\nSample prompt A : {df['prompt_A'].iloc[0]}")
print(f"Sample prompt B : {df['prompt_B'].iloc[0]}")
print(f"Sample prompt C : {df['prompt_C'].iloc[0]}")
print(f"Sample prompt D : {df['prompt_D'].iloc[0]}")

Building prompts for all rows...
Done — prompts built for 5793 rows

Sample prompt A : The image shows a group of women with glasses on their faces. The meme text says Shivam @shivamishraa Girls be named naina and then have eyes that don't work. This meme is <mask>.
Sample prompt B : a group of women with glasses on their faces. Shivam @shivamishraa Girls be named naina and then have eyes that don't work. Overall this is <mask>.
Sample prompt C : The image shows a group of women with glasses on their faces. The sentiment of this image is <mask>.
Sample prompt D : The meme text says Shivam @shivamishraa Girls be named naina and then have eyes that don't work. This meme is <mask>.


In [ ]:
from transformers import AutoTokenizer

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")

def count_tokens(text):
    return len(tokenizer(text)["input_ids"])

df["tokens_A"] = df["prompt_A"].apply(count_tokens)
df["tokens_B"] = df["prompt_B"].apply(count_tokens)
df["tokens_C"] = df["prompt_C"].apply(count_tokens)
df["tokens_D"] = df["prompt_D"].apply(count_tokens)

print("\nTOKEN LENGTH CHECK (limit = 512)")
print("-" * 45)
for t in ["A", "B", "C", "D"]:
    col = f"tokens_{t}"
    print(f"\nTemplate {t}:")
    print(f"  Average : {df[col].mean():.0f} tokens")
    print(f"  Max     : {df[col].max()} tokens")
    print(f"  Over 512: {(df[col] > 512).sum()} rows")

Loading tokenizer...


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]


TOKEN LENGTH CHECK (limit = 512)
---------------------------------------------

Template A:
  Average : 49 tokens
  Max     : 212 tokens
  Over 512: 0 rows

Template B:
  Average : 43 tokens
  Max     : 206 tokens
  Over 512: 0 rows

Template C:
  Average : 26 tokens
  Max     : 95 tokens
  Over 512: 0 rows

Template D:
  Average : 34 tokens
  Max     : 199 tokens
  Over 512: 0 rows


In [ ]:
# Check and fix all four templates

for template, col in [("A","prompt_A"),("B","prompt_B"),
                       ("C","prompt_C"),("D","prompt_D")]:
    over = (df[f"tokens_{template}"] > 512).sum()

    if over > 0:
        print(f"Template {template}: {over} prompts over 512 — fixing...")

        def fix_prompt(row, t=template):
            text = str(row["Img-Text"])
            cap  = row["blip_caption"]
            # Trim meme text if too long
            if len(text) > 200:
                text = text[:200] + "..."
            if t == "A": return build_prompt_A(cap, text)
            if t == "B": return build_prompt_B(cap, text)
            if t == "C": return build_prompt_C(cap, text)
            if t == "D": return build_prompt_D(cap, text)

        mask = df[f"tokens_{template}"] > 512
        df.loc[mask, col] = df[mask].apply(fix_prompt, axis=1)
        print(f"  Fixed ✓")
    else:
        print(f"Template {template}: all within limit ✓")

Template A: all within limit ✓
Template B: all within limit ✓
Template C: all within limit ✓
Template D: all within limit ✓


In [ ]:
save_path = "/content/drive/MyDrive/Multimodal/multibully_with_prompts.csv"
df.to_csv(save_path, index=False)

print(f"Saved to Drive : {save_path}")
print(f"Total rows     : {len(df)}")
print(f"\nNew columns added:")
print(f"  prompt_A — main template (caption + text, structured)")
print(f"  prompt_B — simple template (caption + text, short)")
print(f"  prompt_C — caption only")
print(f"  prompt_D — text only (no caption)")
print(f"\nPhase 3 complete!")
print("Next: Phase 4 — XLM-RoBERTa Classification")

Saved to Drive : /content/drive/MyDrive/Multimodal/multibully_with_prompts.csv
Total rows     : 5793

New columns added:
  prompt_A — main template (caption + text, structured)
  prompt_B — simple template (caption + text, short)
  prompt_C — caption only
  prompt_D — text only (no caption)

Phase 3 complete!
Next: Phase 4 — XLM-RoBERTa Classification
